# Submit cluster-side QC analysis (SLURM)

Run this notebook **on a cluster login/transfer node** (needs `sbatch`/`sacct`
on `PATH`), pointed at wherever your data has landed after transferring it
yourself from the NAS (Globus/FileZilla) -- `SAMPLE_DIR` below is a cluster
path. Partial arrival is completely fine: it only ever acts on files that
already exist, and picks up more the next time it's run.

For each pending round, it:
1. Writes a manifest of that round's not-yet-analysed FOV image paths.
2. Submits a SLURM **array job** running `analysis/cli_analyze_fov.py` once
   per pending FOV (mirrors `FOVScheduler`'s local `analyze_file` call).
3. Once a round's FOVs are all done, submits a job running
   `analysis/cli_build_round_mosaic.py` (mirrors `RoundScheduler`'s local
   `build_round_mosaics` call).

Submission is tracked with `.fov_submitted`/`.round_mosaic_submitted`
sentinels (holding the submitted job id) so re-running this notebook while a
previous array job is still `PENDING`/`RUNNING` (checked via `sacct`) does
**not** resubmit the same work.

No `pip install` needed here either -- `MERci/` just needs to exist as a repo
clone alongside the data (same convention as everywhere else in this repo);
the generated sbatch scripts invoke the CLI scripts by their absolute path
under this same clone.

## 1 — Setup

In [ ]:
import os
import sys
import time
import logging
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root on the cluster
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress        import ProgressTracker
from MERci.acquisition     import cluster_submit

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

Edit to match your experiment and cluster account. Set `DRY_RUN = True` the
first time you run this against a new experiment to see what *would* be
submitted without actually calling `sbatch`.

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

# ── Image file format (must match what HAL wrote) ────────────────────────────
IMAGE_SUFFIX = ".zarr"   # options: ".zarr", ".dax", ".tiff"

# ── SLURM job parameters ──────────────────────────────────────────────────
PARTITION         = "zhuang,sapphire,shared"
CONDA_ENV         = "merci_env"   # cluster-side env with the same scientific deps as environment.yml
ARRAY_CONCURRENCY = 50            # max FOV-analysis array tasks running at once
FOV_MEM,    FOV_TIME    = "8gb",  "02:00:00"
MOSAIC_MEM, MOSAIC_TIME = "8gb",  "00:30:00"

# ── Behaviour ──────────────────────────────────────────────────────────────
DRY_RUN       = True    # True: build scripts + print what would be submitted, but don't call sbatch
POLL_INTERVAL = 1800    # seconds between passes if you run the continuous loop in section 5

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Positions tag: {POSITIONS_TAG}")
print(f"Image suffix : {IMAGE_SUFFIX}")
print(f"Partition    : {PARTITION}")
print(f"Conda env    : {CONDA_ENV}")
print(f"Dry run      : {DRY_RUN}")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

MANIFESTS_DIR = config.analysis_dir / "logs"
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Rounds     : {meta.n_rounds}")
print(f"FOVs       : {meta.n_fovs}")
print(f"Manifests  : {MANIFESTS_DIR}")

## 3 — Check current progress

In [ ]:
summary = tracker.summary(meta)
print(f"FOVs  done : {summary['files_fov_done']} / {summary['files_total']}")
print(f"Rounds done: {summary['rounds_done']} / {summary['rounds_total']}")

## 4 — Submission functions

In [ ]:
def submit_pending_fov_analysis(dry_run: bool = False) -> list:
    """Submit one FOV-analysis array job per round that has pending FOVs and
    no still-active previous submission. Returns [(round_id, job_id, n_fovs), ...]."""
    pending_all = set(tracker.pending_fov_files(meta.all_expected_files()))
    submitted = []
    for rid in meta.valid_round_ids():
        round_files = sorted(f for f in meta.files_for_round(rid) if f in pending_all)
        if not round_files:
            continue

        prev_job = tracker.fov_analysis_submitted_job_id(rid)
        if prev_job is not None and cluster_submit.is_job_active(prev_job):
            print(f"Round {rid}: FOV job {prev_job} still active — skipping.")
            continue

        manifest_path = MANIFESTS_DIR / f"pending_fovs_round{rid:03d}.txt"
        manifest_path.write_text("\n".join(str(f) for f in round_files) + "\n")
        script_path = MANIFESTS_DIR / f"fov_array_round{rid:03d}.sh"
        cluster_submit.build_fov_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, n_pending=len(round_files),
            output_path=script_path, array_concurrency=ARRAY_CONCURRENCY,
            mem=FOV_MEM, time=FOV_TIME, partition=PARTITION, conda_env=CONDA_ENV,
        )
        if dry_run:
            print(f"[dry run] would submit {script_path}  ({len(round_files)} FOV(s))")
            continue
        job_id = cluster_submit.submit_sbatch(script_path)
        if job_id is not None:
            tracker.mark_fov_analysis_submitted(rid, job_id)
            submitted.append((rid, job_id, len(round_files)))
    return submitted


def submit_pending_round_mosaics(dry_run: bool = False) -> list:
    """Submit one job (array if >1 round) building mosaics for every round
    whose FOVs are all done but has no mosaic yet and no still-active
    previous submission. Returns [(round_ids, job_id)] or []."""
    pending_rounds = [
        rid for rid in tracker.pending_rounds(meta.valid_round_ids(), meta, config.fov_subset)
        if not (
            tracker.is_round_mosaic_submitted(rid)
            and (job_id := tracker.round_mosaic_submitted_job_id(rid)) is not None
            and cluster_submit.is_job_active(job_id)
        )
    ]
    if not pending_rounds:
        return []

    manifest_path = MANIFESTS_DIR / "pending_rounds.txt"
    manifest_path.write_text("\n".join(str(r) for r in pending_rounds) + "\n")
    script_path = MANIFESTS_DIR / "round_mosaic.sh"
    cluster_submit.build_round_mosaic_script(
        sample_dir=SAMPLE_DIR, manifest_path=manifest_path, n_pending=len(pending_rounds),
        output_path=script_path, mem=MOSAIC_MEM, time=MOSAIC_TIME,
        partition=PARTITION, conda_env=CONDA_ENV,
    )
    if dry_run:
        print(f"[dry run] would submit {script_path}  (rounds {pending_rounds})")
        return []
    job_id = cluster_submit.submit_sbatch(script_path)
    if job_id is None:
        return []
    for rid in pending_rounds:
        tracker.mark_round_mosaic_submitted(rid, job_id)
    return [(pending_rounds, job_id)]

## 5 — Run one pass

Re-run this cell by hand each time you've pulled more data over from the NAS
(Globus/FileZilla) — it only ever acts on files that already exist, so it's
safe to call as often as you like.

In [ ]:
def run_once(dry_run: bool = DRY_RUN) -> None:
    fov_submitted    = submit_pending_fov_analysis(dry_run=dry_run)
    mosaic_submitted = submit_pending_round_mosaics(dry_run=dry_run)
    summary = tracker.summary(meta)
    print(f"FOV job(s) submitted this pass   : {fov_submitted}")
    print(f"Mosaic job(s) submitted this pass: {mosaic_submitted}")
    print(f"FOVs  done : {summary['files_fov_done']} / {summary['files_total']}")
    print(f"Rounds done: {summary['rounds_done']} / {summary['rounds_total']}")

run_once()

## 6 — Optional: run continuously

Only run this cell if you'd rather leave the notebook polling than re-run
section 5 by hand — a login node tolerates a low-frequency Python poll fine.
Background it with `tmux`/`nohup` if you close the notebook. Interrupt the
kernel (`■` button) to stop.

In [ ]:
iteration = 0
while True:
    print(f"=== tick {iteration} ===")
    run_once()
    iteration += 1
    time.sleep(POLL_INTERVAL)